<a href="https://colab.research.google.com/github/gautamkr1876/AIML_ClassNotes/blob/main/7.%20Advanced%20AI%20Agents/9.Model%20Quantization%20Techniques/model_quantization.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [3]:
round(-1/(7/255)) + 36

0

In [4]:
import numpy as np

# Suppress scientific notation
np.set_printoptions(suppress=True)

# Generate randomly distributed parameters
params = np.random.uniform(low=-50, high=150, size=20)

# Make sure important values are at the beginning for better debugging
params[0] = params.max() + 1
params[1] = params.min() - 1
params[2] = 0

# Round each number to the second decimal place
params = np.round(params, 2)

# Print the parameters
print(params)

[129.95 -49.89   0.    61.54 104.71  24.77 -43.81 102.18 115.99   9.43
  60.87 -18.47  50.51  59.45  74.49  69.66  42.05  25.46  24.33 128.95]


In [6]:
def clamp(params_q: np.array, lower_bound: int, upper_bound: int) -> np.array:
    params_q[params_q < lower_bound] = lower_bound
    params_q[params_q > upper_bound] = upper_bound
    return params_q


def asymmetric_quantization(params: np.array, bits: int) -> tuple[np.array, float, int]:
    # Calculate the scale and zero point
    alpha = np.max(params)
    beta = np.min(params)
    scale = (alpha - beta) / (2**bits-1)
    zero = -1*np.round(beta / scale)
    lower_bound, upper_bound = 0, 2**bits-1
    # Quantize the parameters
    quantized = clamp(np.round(params / scale + zero), lower_bound, upper_bound).astype(np.int32)
    return quantized, scale, zero

def asymmetric_dequantize(params_q: np.array, scale: float, zero: int) -> np.array:
    return (params_q - zero) * scale

def quantization_error(params: np.array, params_q: np.array):
    # calculate the MSE
    return np.mean((params - params_q)**2)

(asymmetric_q, asymmetric_scale, asymmetric_zero) = asymmetric_quantization(params, 8)


print(f'Original:')
print(np.round(params, 2))
print('')
print(f'Asymmetric scale: {asymmetric_scale}, zero: {asymmetric_zero}')
print(asymmetric_q)


Original:
[129.95 -49.89   0.    61.54 104.71  24.77 -43.81 102.18 115.99   9.43
  60.87 -18.47  50.51  59.45  74.49  69.66  42.05  25.46  24.33 128.95]

Asymmetric scale: 0.7052549019607842, zero: 71.0
[255   0  71 158 219 106   9 216 235  84 157  45 143 155 177 170 131 107
 105 254]


In [7]:
# Dequantize the parameters back to 32 bits
params_deq_asymmetric = asymmetric_dequantize(asymmetric_q, asymmetric_scale, asymmetric_zero)

print(f'{"Asymmetric error: ":>20}{np.round(quantization_error(params, params_deq_asymmetric), 2)}')


  Asymmetric error: 0.04


In [9]:
print(f'{"Asymmetric dequantized: ":>20}{np.round(params_deq_asymmetric, 2)}')
print(f'{"Original: ":>20}{np.round(params, 2)}')


Asymmetric dequantized: [129.77 -50.07   0.    61.36 104.38  24.68 -43.73 102.26 115.66   9.17
  60.65 -18.34  50.78  59.24  74.76  69.82  42.32  25.39  23.98 129.06]
          Original: [129.95 -49.89   0.    61.54 104.71  24.77 -43.81 102.18 115.99   9.43
  60.87 -18.47  50.51  59.45  74.49  69.66  42.05  25.46  24.33 128.95]
